In [1]:
! pip install nltk  scikit-learn pandas

In [7]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

import re
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
data = {
    "text": [
        "I love Natural Language Processing!",
        "Machine learning is very interesting.",
        "I hate errors in my code.",
        "Python is amazing for AI."
    ],
    "label": ["positive", "positive", "negative", "positive"]
}

df = pd.DataFrame(data)
df


,text,label
0,I love Natural Language Processing!,positive
1,Machine learning is very interesting.,positive
2,I hate errors in my code.,negative
3,Python is amazing for AI.,positive


In [9]:
#text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)   # remove punctuation & numbers
    return text


In [10]:
#lemmatization + stopword removal
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)


In [11]:
#apply cleaning
df["clean_text"] = df["text"].apply(clean_text)
df["processed_text"] = df["clean_text"].apply(preprocess)

df


,text,label,clean_text,processed_text
0,I love Natural Language Processing!,positive,i love natural language processing,love natural language processing
1,Machine learning is very interesting.,positive,machine learning is very interesting,machine learning interesting
2,I hate errors in my code.,negative,i hate errors in my code,hate error code
3,Python is amazing for AI.,positive,python is amazing for ai,python amazing ai


In [12]:
# label encoding
encoder = LabelEncoder()
df["encoded_label"] = encoder.fit_transform(df["label"])

df


,text,label,clean_text,processed_text,encoded_label
0,I love Natural Language Processing!,positive,i love natural language processing,love natural language processing,1
1,Machine learning is very interesting.,positive,machine learning is very interesting,machine learning interesting,1
2,I hate errors in my code.,negative,i hate errors in my code,hate error code,0
3,Python is amazing for AI.,positive,python is amazing for ai,python amazing ai,1


In [13]:
# TF-IDF representation
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["processed_text"])

In [14]:
# TF-IDF to Dataframe
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df

,ai,amazing,code,error,hate,interesting,language,learning,love,machine,natural,processing,python
0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.5,0.00000,0.5,0.00000,0.5,0.5,0.00000
1,0.00000,0.00000,0.00000,0.00000,0.00000,0.57735,0.0,0.57735,0.0,0.57735,0.0,0.0,0.00000
2,0.00000,0.00000,0.57735,0.57735,0.57735,0.00000,0.0,0.00000,0.0,0.00000,0.0,0.0,0.00000
3,0.57735,0.57735,0.00000,0.00000,0.00000,0.00000,0.0,0.00000,0.0,0.00000,0.0,0.0,0.57735


In [15]:
#Save outputs
df.to_csv("cleaned_text_output.csv", index=False)
tfidf_df.to_csv("tfidf_vectors.csv", index=False)